<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/Cohort_9_PineconeIndexDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pinecone Vector Store

If you're opening this Notebook on colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install llama-index llama-index-vector-stores-pinecone

In [11]:
import logging
import sys
import os

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

#### Creating a Pinecone Index

In [12]:
from pinecone import Pinecone, ServerlessSpec

In [19]:
import openai
from google.colab import userdata

# Retrieve the OpenAI API key from Google Colab secrets
openai.api_key = userdata.get('openai')

#if openai.api_key:
#    os.environ["OPENAI_API_KEY"] = openai.api_key

In [23]:
import openai
from google.colab import userdata

# Retrieve the OpenAI API key from Google Colab secrets
api_key = userdata.get('PINECONE_API_KEY')

#api_key = os.environ["PINECONE_API_KEY"]

pc = Pinecone(api_key=api_key)

In [15]:

#delete if needed 9 first time no need to delete)
pc.delete_index("quickstart")

In [ ]:
# dimensions are for text-embedding-ada-002

pc.create_index(
    name="quickstart",
    dimension=1536,
    metric="euclidean",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)


In [26]:
pinecone_index = pc.Index("quickstart")

#### Load documents, build the PineconeVectorStore and VectorStoreIndex

In [27]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.vector_stores.pinecone import PineconeVectorStore
from IPython.display import Markdown, display

Download Data

In [39]:
import os
from google.colab import files
import shutil

# Define the target directory
target_directory = './data/'

# Create the target directory if it doesn't exist
os.makedirs(target_directory, exist_ok=True)

print("Please select the file you want to upload:")
uploaded = files.upload()

for filename in uploaded.keys():
  # Define the destination path for the uploaded file
  destination_path = os.path.join(target_directory, filename)

  # Move the uploaded file from the current directory to the target directory
  shutil.move(filename, destination_path)
  print(f"File '{filename}' successfully uploaded and moved to '{target_directory}'")


Please select the file you want to upload:


Saving DDS_Employee_Handbook_Synthetic_v1 (1).pdf to DDS_Employee_Handbook_Synthetic_v1 (1).pdf
Saving DDS_HR_FAQ_Synthetic_v1.pdf to DDS_HR_FAQ_Synthetic_v1.pdf
Saving DDS_Remote_Work_Policy_Synthetic_v1.pdf to DDS_Remote_Work_Policy_Synthetic_v1.pdf
Saving DDS_Leave_Policy_Synthetic_v1.pdf to DDS_Leave_Policy_Synthetic_v1.pdf
File 'DDS_Employee_Handbook_Synthetic_v1 (1).pdf' successfully uploaded and moved to './data/'
File 'DDS_HR_FAQ_Synthetic_v1.pdf' successfully uploaded and moved to './data/'
File 'DDS_Remote_Work_Policy_Synthetic_v1.pdf' successfully uploaded and moved to './data/'
File 'DDS_Leave_Policy_Synthetic_v1.pdf' successfully uploaded and moved to './data/'


In [40]:
# load documents
documents = SimpleDirectoryReader("./data/").load_data()

In [41]:
documents

[Document(id_='a95baa34-9724-4a9a-a335-cd762f2e3c5e', embedding=None, metadata={'page_label': '1', 'file_name': 'DDS_Employee_Handbook_Synthetic_v1 (1).pdf', 'file_path': '/content/data/DDS_Employee_Handbook_Synthetic_v1 (1).pdf', 'file_type': 'application/pdf', 'file_size': 9665, 'creation_date': '2026-03-14', 'last_modified_date': '2026-03-14'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='DDS Employee Handbook (Synthetic) v1\nEffective date: March 03, 2026 \x7f Dubai (GST)\nNote: This document is a synthetic, training-friendly employee handbook for demos, onboarding\nsimulations, and HR-policy chatbot prototypes. It is not legal advi

In [ ]:
# initialize without metadata filter
from llama_index.core import StorageContext

vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)

#### Query Index

May take a minute or so for the index to be ready!

In [ ]:
# set Logging to DEBUG for more detailed outputs
query_engine = index.as_query_engine()
response = query_engine.query("How is confirmation handled after probation?")
display(Markdown(f"<b>{response}</b>"))

In [ ]:
pip install gradio

In [ ]:
# Install necessary packages
#%pip install -q llama-index llama-index-vector-stores-pinecone gradio

# --- Imports ---
import os
import logging
import sys
import gradio as gr
from IPython.display import Markdown, display

from pinecone import Pinecone, ServerlessSpec
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.vector_stores.pinecone import PineconeVectorStore

# --- Logging ---
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

# --- Set API Keys ---

PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")



api_key = os.environ["PINECONE_API_KEY"]


# --- Initialize Pinecone ---
pc = Pinecone(api_key=api_key)
index_name = "quickstart"
dimension = 1536

# Delete index if exists (optional)
if index_name in [idx['name'] for idx in pc.list_indexes()]:
    pc.delete_index(index_name)

# Create Pinecone index
pc.create_index(
    name=index_name,
    dimension=dimension,
    metric="euclidean",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

pinecone_index = pc.Index(index_name)



#change directory
documents = SimpleDirectoryReader("./data/").load_data()

# --- Create Index ---
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

# --- Query Engine ---
query_engine = index.as_query_engine()

# --- Gradio App ---
def query_doc(prompt):
    try:
        response = query_engine.query(prompt)
        return str(response)
    except Exception as e:
        return f"Error: {str(e)}"

gr.Interface(
    fn=query_doc,
    inputs=gr.Textbox(label="Ask a question about the document"),
    outputs=gr.Textbox(label="Answer"),
    title="DDS Enterprise chatbot",
    description="Ask questions based on the indexed Social Media Regualtion. Powered by LlamaIndex & Pinecone."
).launch(share=True)


In [ ]:
# Install necessary packages
# %pip install -q llama-index llama-index-vector-stores-pinecone gradio pinecone

# --- Imports ---
import os
import logging
import sys
import gradio as gr
from IPython.display import Markdown, display

from pinecone import Pinecone, ServerlessSpec
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.vector_stores.pinecone import PineconeVectorStore

# --- Logging ---
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

# --- Set API Keys ---
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
api_key = os.environ["PINECONE_API_KEY"]

# --- Initialize Pinecone ---
pc = Pinecone(api_key=api_key)
index_name = "quickstart"
dimension = 1536

# Delete index if exists (optional)
if index_name in [idx["name"] for idx in pc.list_indexes()]:
    pc.delete_index(index_name)

# Create Pinecone index
pc.create_index(
    name=index_name,
    dimension=dimension,
    metric="euclidean",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

pinecone_index = pc.Index(index_name)

# --- Load Documents ---
documents = SimpleDirectoryReader("./data/").load_data()

# --- Create Index ---
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

# --- Query Engine ---
query_engine = index.as_query_engine()

# --- Query Function ---
def query_doc(prompt):
    try:
        response = query_engine.query(prompt)
        return str(response)
    except Exception as e:
        return f"Error: {str(e)}"

# --- Sample Questions ---
def fill_example(question):
    return question

# --- Custom CSS ---
custom_css = """
.gradio-container {
    max-width: 1200px !important;
    margin: 0 auto !important;
    font-family: Arial, sans-serif;
}
.main-title {
    text-align: center;
    font-size: 30px;
    font-weight: 700;
    margin-bottom: 8px;
}
.sub-title {
    text-align: center;
    font-size: 16px;
    color: #555;
    margin-bottom: 20px;
}
.info-box {
    border: 1px solid #e5e7eb;
    border-radius: 14px;
    padding: 18px;
    background: #f9fafb;
}
"""

# --- Gradio App with Blocks ---
with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
    # Header
    gr.HTML(
        """
        <div style="text-align:center; padding-top: 10px;">
            <img src="https://raw.githubusercontent.com/Decoding-Data-Science/airesidency/main/dds_logo.jpg"
                 alt="DDS Logo"
                 style="height:90px; border-radius:12px; margin-bottom:10px;">
            <div class="main-title">DDS Enterprise Chatbot</div>
            <div class="sub-title">
                Ask questions based on the indexed Social Media Regulation.<br>
                Powered by LlamaIndex, Pinecone, and Gradio.
            </div>
        </div>
        """
    )

    with gr.Row(equal_height=True):
        # Left Column
        with gr.Column(scale=1):
            gr.Markdown("### Ask Your Question")

            user_input = gr.Textbox(
                label="Question",
                placeholder="Type your question here...",
                lines=5
            )

            with gr.Row():
                ask_btn = gr.Button("Ask", variant="primary")
                clear_btn = gr.Button("Clear")

            gr.Markdown("### Suggested Questions")
            ex1 = gr.Button("What is this document about?")
            ex2 = gr.Button("Summarize the key points.")
            ex3 = gr.Button("What are the important regulations mentioned?")
            ex4 = gr.Button("Who is this regulation relevant for?")

        # Right Column
        with gr.Column(scale=1):
            gr.Markdown("### Response")
            output = gr.Textbox(
                label="Answer",
                lines=18,
                interactive=False,
                show_copy_button=True
            )

            gr.HTML(
                """
                <div class="info-box" style="margin-top: 12px;">
                    <b>About this app</b><br>
                    This assistant searches your indexed documents and returns answers using
                    vector search with Pinecone and LlamaIndex.
                </div>
                """
            )

    # Actions
    ask_btn.click(fn=query_doc, inputs=user_input, outputs=output)
    user_input.submit(fn=query_doc, inputs=user_input, outputs=output)

    clear_btn.click(fn=lambda: ("", ""), inputs=[], outputs=[user_input, output])

    ex1.click(fn=fill_example, inputs=gr.State("What is this document about?"), outputs=user_input)
    ex2.click(fn=fill_example, inputs=gr.State("Summarize the key points."), outputs=user_input)
    ex3.click(fn=fill_example, inputs=gr.State("What are the important regulations mentioned?"), outputs=user_input)
    ex4.click(fn=fill_example, inputs=gr.State("Who is this regulation relevant for?"), outputs=user_input)

demo.launch(share=True)